In [1]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import sys
from pathlib import Path

import torch

sys.path.append("../src")

from hand_to_tex.datasets.dataloader import HMEDataLoaderFactory
from hand_to_tex.models import BaselineTransformer
from hand_to_tex.training import BaselineTrainer
from hand_to_tex.utils import LatexVocab

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"We are running isolated processes on: {device}")

We are running isolated processes on: mps


## 1. Vocabulary and DataLoaders


In [2]:
vocab = LatexVocab.load(Path("../data/assets/vocab.json"))
print(f"Loaded Vocabulary Dimension Limits: {len(vocab._encode)} items")

root_path = Path("../data/mathwriting-2024")

factory = HMEDataLoaderFactory(root=root_path, processed=True, vocab=vocab, batch_size=32)
train_loader = factory.train()

try:
    valid_loader = factory.valid()
except Exception:
    print("No validation set found. Reusing train_loader for validation tests...")
    valid_loader = factory.train()

print(f"Batches established successfully: {len(train_loader)} separate batches.")

Loaded Vocabulary Dimension Limits: 258 items
Batches established successfully: 312 separate batches.


## 2. Model Initialization

In [3]:
model = BaselineTransformer(
    vocab_size=len(vocab._encode),
    pad_idx=vocab.PAD,
    d_model=64,  # Dimensional size of the abstract mathematical thought state
    nhead=2,  # Number of parallel attention heads running logic checks
    num_encoder_layers=2,  # Transformer blocks decoding the geometric physical handwriting
    num_decoder_layers=2,  # Transformer blocks translating geometry to math text sequence
    in_features=10,  # Initial parameters per point (speed, coordinates, etc.)
)

print(
    f"Model Trainable Constraint count: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters"
)

Model Trainable Constraint count: 666,562 parameters


## 3. Training Loop Execution

In [4]:
trainer = BaselineTrainer(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    vocab=vocab,
    device=device,
    lr=5e-5,
)


trainer.train(num_epochs=10)

========== Training for 10 epochs ==========

[ Epoch 1/10 ]


/Users/marcin/Documents/GitHub/hand-to-tex/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/marcin/Documents/GitHub/hand-to-tex/.venv/lib/python3.12/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


   [Batch 100/312] Loss: 4.4307
   [Batch 200/312] Loss: 4.0943
   [Batch 300/312] Loss: 3.8450


/Users/marcin/Documents/GitHub/hand-to-tex/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:482: UserWarning: The operator 'aten::_nested_tensor_from_mask_left_aligned' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:34.)
  elif do_mask_check and not torch._nested_tensor_from_mask_left_aligned(


Epoch completed in 82.28s
Train Loss: 4.3195  |  Valid Loss: 3.9429

[ Epoch 2/10 ]
   [Batch 100/312] Loss: 3.7258
   [Batch 200/312] Loss: 3.6099
   [Batch 300/312] Loss: 3.3640
Epoch completed in 70.26s
Train Loss: 3.5993  |  Valid Loss: 3.5718

[ Epoch 3/10 ]
   [Batch 100/312] Loss: 3.3032
   [Batch 200/312] Loss: 3.3704
   [Batch 300/312] Loss: 3.1986
Epoch completed in 69.80s
Train Loss: 3.3521  |  Valid Loss: 3.3766

[ Epoch 4/10 ]
   [Batch 100/312] Loss: 3.3057
   [Batch 200/312] Loss: 3.4726
   [Batch 300/312] Loss: 3.0543
Epoch completed in 70.68s
Train Loss: 3.2181  |  Valid Loss: 3.2547

[ Epoch 5/10 ]
   [Batch 100/312] Loss: 3.0548
   [Batch 200/312] Loss: 2.9810
   [Batch 300/312] Loss: 3.0946
Epoch completed in 67.71s
Train Loss: 3.1298  |  Valid Loss: 3.1720

[ Epoch 6/10 ]
   [Batch 100/312] Loss: 3.2279
   [Batch 200/312] Loss: 2.9022
   [Batch 300/312] Loss: 2.9396
Epoch completed in 69.20s
Train Loss: 3.0645  |  Valid Loss: 3.1045

[ Epoch 7/10 ]
   [Batch 100/31

In [ ]:
# model.eval()

# batch = next(iter(valid_loader))
# padded_ft, ft_lengths, padded_ts, ts_lengths = batch
# padded_ft = padded_ft.to(device)
# padded_ts = padded_ts.to(device)

# # Pick the first handwriting sample in the batch
# idx = 0
# sample_ft = padded_ft[idx:idx+1]
# sample_ft_len = [ft_lengths[idx]]

# with torch.no_grad():
#     sos_idx = vocab.SOS  # Usually 1
#     eos_idx = vocab.EOS  # Usually 2

#     # Initialize the generated sequence with `<sos>`
#     generated = torch.tensor([[sos_idx]], dtype=torch.long, device=device)

#     # Autoregressive generation loop
#     for _ in range(50):
#         # Forward pass the sequence generated so far
#         output = model(src=sample_ft, src_lengths=sample_ft_len, tgt=generated)

#         # Grab the logits for the NEXT token prediction
#         next_token_idx = output[0, -1, :].argmax(dim=-1).item()

#         # Append token dynamically
#         next_token_tensor = torch.tensor([[next_token_idx]], dtype=torch.long, device=device)
#         generated = torch.cat([generated, next_token_tensor], dim=1)

#         if next_token_idx == eos_idx:
#             break

# # Decode back into readable strings!
# print("=== GREEDY INFERENCE DECODING ===")

# ground_truth = padded_ts[idx].tolist()
# predicted = generated[0].tolist()

# print(f"\nGround Truth LaTeX:")
# print(" ".join(vocab.decode_sequence(ground_truth)))

# print(f"\nPredicted LaTeX:")
# print(" ".join(vocab.decode_sequence(predicted)))